## 🎯 Learning Objectives
* Understand the limitations of traditional NLP metrics for evaluating RAG systems.
* Learn the core concepts and metrics provided by RAGAs for RAG-specific evaluation.
* Explore how TruLens provides observability, tracing, and evaluation for RAG applications.
* Implement and interpret evaluation results using RAGAs and TruLens on a sample RAG system.
* Identify appropriate use cases and performance considerations for each evaluation framework.


## Evaluating Production-Ready RAG Systems: RAGAs and TruLens

As we move from experimental RAG prototypes to production-grade systems, robust evaluation becomes paramount. A RAG system's performance isn't just about the final answer; it's about the quality of the retrieved context, the faithfulness of the generation to that context, and the overall relevance to the user's query. Traditional NLP metrics like BLEU or ROUGE, while useful for summarization or translation, fall short here because they don't account for the *retrieval* component or the *groundedness* of the answer in the provided context.

Imagine you're building a self-driving car. You wouldn't just evaluate if the car reached its destination; you'd also evaluate if it followed traffic laws, avoided obstacles, and maintained a safe distance. Similarly, for RAG, we need to evaluate not just the final answer, but also the intermediate steps: the quality of the retrieved information and how well the LLM utilized it.

This is where specialized RAG evaluation frameworks like **RAGAs** and **TruLens** come into play. They provide a structured way to assess different aspects of a RAG system, often leveraging the power of LLMs themselves as 'judges' to score various metrics.

### RAGAs: RAG-specific Metrics for Granular Evaluation

RAGAs (Retrieval Augmented Generation Assessment) is a framework designed specifically to evaluate the quality of RAG systems. It focuses on metrics that are crucial for RAG, such as:

*   **Faithfulness**: How much of the generated answer is directly supported by the retrieved context? (Crucial for preventing hallucinations).
*   **Answer Relevance**: Is the generated answer directly relevant to the user's question?
*   **Context Relevance**: Is the retrieved context relevant to the user's question?
*   **Context Recall**: How much of the ground truth answer is covered by the retrieved context?
*   **Answer Correctness**: How accurate is the generated answer compared to a ground truth?

RAGAs uses an LLM to act as an evaluator, providing scores for these metrics. This allows for a more nuanced and human-like assessment than purely lexical overlap metrics.

### TruLens: Observability, Tracing, and Evaluation for LLM Apps

TruLens takes a broader approach, offering a comprehensive platform for observability, tracing, and evaluation of LLM-powered applications, including RAG systems. While RAGAs focuses purely on RAG-specific metrics, TruLens provides:

*   **End-to-end Tracing**: Visualize the flow of data through your RAG pipeline, from query to retrieval to generation.
*   **Feedback Functions**: Define custom evaluation criteria (often using LLMs) to score various aspects of your application's output, similar to RAGAs metrics but with more flexibility.
*   **Dashboard**: A UI to explore traces, aggregate evaluation scores, and debug issues.

Think of TruLens as a flight recorder for your RAG system. It captures every step, allowing you to replay and analyze exactly what happened, where things went wrong, and how well each component performed. It's particularly powerful for identifying bottlenecks or failure points in complex RAG chains.

Both frameworks are essential tools in the RAG engineer's toolkit for building reliable, high-performing, and production-ready RAG applications.


In [ ]:
# Install necessary libraries (as of 2026, these versions are stable and widely used)
# !pip install -q llama-index==0.12.0 openai==1.30.0 ragas==0.1.0 trulens_eval==0.20.0

import os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.embeddings import resolve_embed_model
from llama_index.llms.openai import OpenAI

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevance, context_relevance
from datasets import Dataset

from trulens_eval import TruLlama, Feedback, Select
from trulens_eval.feedback import Groundedness
from trulens_eval.feedback.provider.openai import OpenAI as TruOpenAI

# --- Configuration --- 
# Set your OpenAI API key. In a real production environment, use environment variables.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Ensure API key is set
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable not set. Please set it to run the code.")

# Configure LlamaIndex settings for consistency
Settings.llm = OpenAI(model="gpt-4o-mini") # Using a cost-effective model for evaluation
Settings.embed_model = resolve_embed_model("text-embedding-3-small")
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print("LlamaIndex settings configured.")

# --- 1. Setup a Simple RAG System --- 

# Create dummy documents for demonstration
dummy_docs_content = [
    "The capital of France is Paris. Paris is known for its Eiffel Tower and Louvre Museum.",
    "The Amazon rainforest is the largest tropical rainforest in the world, spanning several South American countries.",
    "Artificial intelligence (AI) is a rapidly evolving field. Machine learning is a subset of AI.",
    "Quantum computing harnesses quantum-mechanical phenomena like superposition and entanglement to perform computations.",
    "The human heart has four chambers: two atria and two ventricles. It pumps blood throughout the body."
]

# In a real scenario, you'd load from files: SimpleDirectoryReader("data").load_data()
from llama_index.core.schema import Document
documents = [Document(text=t) for t in dummy_docs_content]

# Create an index
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=2)

print("Simple RAG system (LlamaIndex query engine) initialized.")

# --- 2. Evaluate with RAGAs --- 

# Define a small dataset for RAGAs evaluation
# In a real scenario, this would be a larger, more diverse dataset.
questions = [
    "What is the capital of France?",
    "Tell me about the Amazon rainforest.",
    "What is AI?",
    "How many chambers does the human heart have?"
]

ground_truths = [
    ["Paris is the capital of France."],
    ["The Amazon rainforest is the largest tropical rainforest globally, located in South America."],
    ["AI is a field of computer science focused on creating intelligent machines. Machine learning is a part of it."],
    ["The human heart has four chambers: two atria and two ventricles."]
]

# Generate answers and retrieve contexts using our RAG system
data_samples = []
for i, q in enumerate(questions):
    response = query_engine.query(q)
    data_samples.append({
        "question": q,
        "answer": str(response),
        "contexts": [node.get_content() for node in response.source_nodes],
        "ground_truths": ground_truths[i]
    })

# Convert to RAGAs Dataset format
raga_dataset = Dataset.from_list(data_samples)

print("RAGAs dataset prepared. Running RAGAs evaluation...")

# Define RAGAs metrics to evaluate
metrics = [
    faithfulness,
    answer_relevance,
    context_relevance
    # context_recall, # Requires ground_truth_contexts, which we don't have for this simple example
    # answer_correctness, # Requires ground_truth_answers, which we have but let's keep it simple
]

# Run evaluation
# Note: This will make several LLM calls, so it might take a moment.
raga_results = evaluate(raga_dataset, metrics=metrics)

print("\n--- RAGAs Evaluation Results ---")
print(raga_results)
print(raga_results.to_pandas())

# --- 3. Evaluate with TruLens --- 

# Initialize TruLens provider for feedback functions
tru_openai = TruOpenAI()

# Define feedback functions
# Groundedness: How much of the answer is supported by the context?
f_groundedness = Feedback(tru_openai.groundedness_measure_with_cot_reasons).on(
    Select.rag.retrieve.docs().node_contents(), 
    Select.rag.llm.completion()
).aggregate(Groundedness.aggregate_of_selection).name("Groundedness")

# Answer Relevance: Is the answer relevant to the question?
f_answer_relevance = Feedback(tru_openai.relevance_with_cot_reasons).on(
    Select.rag.query(), 
    Select.rag.llm.completion()
).name("Answer Relevance")

# Context Relevance: Is the retrieved context relevant to the question?
f_context_relevance = Feedback(tru_openai.relevance_with_cot_reasons).on(
    Select.rag.query(), 
    Select.rag.retrieve.docs().node_contents()
).aggregate(Groundedness.aggregate_of_selection).name("Context Relevance")

# Initialize TruLens recorder for our LlamaIndex query engine
tru_recorder = TruLlama(
    query_engine, 
    app_id="LlamaIndex_RAG_App",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance]
)

print("\nTruLens recorder initialized. Running queries through TruLens...")

# Run queries through the TruLens-wrapped query engine
with tru_recorder as recording:
    for q in questions:
        response = query_engine.query(q)
        print(f"Query: {q}\nResponse: {response.response}\n")

print("\n--- TruLens Evaluation Results (Summary) ---")
# Get evaluation records
from trulens_eval import Tru
tru = Tru()
records = tru.get_records_and_feedback(app_ids=["LlamaIndex_RAG_App"])

# Display average scores
if not records.empty:
    print(records[['input', 'output', 'Groundedness', 'Answer Relevance', 'Context Relevance']].head())
    print("\nAverage Feedback Scores:")
    print(records[['Groundedness', 'Answer Relevance', 'Context Relevance']].mean())
else:
    print("No records found. Ensure the TruLens dashboard is running or check for errors.")

print("\nTo view detailed traces and a dashboard, run: `tru.run_dashboard()` in a separate cell or terminal.")


### Interpreting the Results and Use Cases

#### RAGAs Output Interpretation

The RAGAs output provides average scores for each metric across your evaluation dataset. Each score typically ranges from 0 to 1, where 1 indicates perfect performance.

*   **Faithfulness**: A low score here suggests that your LLM is generating information not present in the retrieved context, indicating potential hallucinations. This is critical for applications requiring high factual accuracy.
*   **Answer Relevance**: A low score means the generated answers are often off-topic or don't directly address the user's question, even if factually correct. This impacts user experience.
*   **Context Relevance**: If this score is low, your retrieval mechanism is fetching irrelevant documents. This can confuse the LLM, increase token usage, and dilute the quality of the final answer.

By analyzing these scores, you can pinpoint specific weaknesses in your RAG system. For example, if `context_relevance` is high but `faithfulness` is low, your retriever is good, but your generator might be hallucinating. If `context_relevance` is low, you might need to improve your embedding model, chunking strategy, or vector store configuration.

#### TruLens Output Interpretation

TruLens provides both aggregated scores and detailed traces for each query. The `get_records_and_feedback` function gives you a DataFrame with inputs, outputs, and scores for each feedback function you defined. The average scores provide an overall health check, similar to RAGAs.

However, the real power of TruLens lies in its **dashboard** (accessible via `tru.run_dashboard()`). The dashboard allows you to:

*   **Visualize Traces**: See the exact sequence of operations for each query, including the raw query, retrieved documents, LLM prompts, and final completion. This is invaluable for debugging.
*   **Inspect Feedback Scores per Query**: Understand why a particular query received a low score by examining the context and answer side-by-side.
*   **Compare Runs**: Track performance over time or across different RAG configurations.

For instance, if a query gets a low `Groundedness` score, you can dive into its trace, examine the retrieved context and the generated answer, and often immediately see where the LLM deviated or hallucinated. If `Context Relevance` is low, you can inspect the retrieved documents to understand why the retriever failed.

#### Performance Trade-offs

Both RAGAs and TruLens heavily rely on LLMs (often powerful models like GPT-4 or Claude Opus) to act as evaluators. This comes with significant trade-offs:

*   **Cost**: Each evaluation metric often requires one or more LLM calls per data point. For large datasets, this can become expensive.
*   **Speed**: LLM calls are inherently slower than traditional metrics. Running a full evaluation suite can take considerable time.
*   **Determinism**: LLM-based evaluations can sometimes exhibit variability, though frameworks like RAGAs and TruLens employ techniques (e.g., Chain-of-Thought prompting for evaluators) to improve consistency.

**Best Practices:**

*   **Sampling**: For large datasets, evaluate on a representative sample rather than the entire dataset.
*   **Targeted Metrics**: Choose only the most critical metrics for your use case to reduce LLM calls.
*   **Offline Evaluation**: Integrate these frameworks into your CI/CD pipeline for automated, offline evaluation before deployment.
*   **Human-in-the-Loop**: Use LLM-based evaluations to flag problematic cases for human review, combining scalability with accuracy.

#### Typical Use Cases

*   **Benchmarking**: Compare different RAG architectures, embedding models, chunking strategies, or LLMs.
*   **Regression Testing**: Ensure that new code changes or data updates don't degrade RAG performance.
*   **Continuous Improvement**: Identify areas for optimization and track progress over time.
*   **Debugging**: Quickly diagnose why a RAG system is failing for specific types of queries.
*   **Production Monitoring**: Integrate with observability platforms to continuously monitor RAG health in live environments.


### Resources

*   **RAGAs Official Documentation**: [https://docs.ragas.io/](https://docs.ragas.io/)
*   **TruLens Official Documentation**: [https://www.trulens.org/](https://www.trulens.org/)
*   **LlamaIndex Documentation**: [https://docs.llamaindex.ai/en/stable/](https://docs.llamaindex.ai/en/stable/)
*   **OpenAI API Documentation**: [https://platform.openai.com/docs/](https://platform.openai.com/docs/)
*   **Paper on RAG Evaluation (e.g., RAGAs research paper)**: Search for "RAGAs: A Framework for Automated Evaluation of Retrieval Augmented Generation Systems" on arXiv or Google Scholar for deeper insights into the methodology.
